## Import libraries

In [26]:
import pandas as pd
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
from statsmodels.stats.proportion import proportions_ztest

umbral = 0.05

In [2]:
df = pd.read_csv('data/ab_data.csv')
print(f"Initial rows and columns: {df.shape}")

#users per group
print(f"    Users per group\n{df.group.value_counts()}\n")

#Detect inconsistencys

df_verification_inconistencys = df.groupby(['group','landing_page'])['user_id'].count().reset_index()
print(f"    Verification users\n{df_verification_inconistencys}\n")

#Cleaning users
#Rules: control == old_page & treatment == new_page

df = df[((df['group']=='control')&(df['landing_page']=='old_page'))
        | ((df['group']=='treatment')&(df['landing_page']=='new_page'))]

#Verification of duplicated users
df = df.drop_duplicates(subset='user_id')
print(f"dups users: {df.user_id.duplicated().sum()}")

print(f"\nRows and cols after cleaning: {df.shape}")
df.head(5)

Initial rows and columns: (294478, 5)
    Users per group
group
treatment    147276
control      147202
Name: count, dtype: int64

    Verification users
       group landing_page  user_id
0    control     new_page     1928
1    control     old_page   145274
2  treatment     new_page   145311
3  treatment     old_page     1965

dups users: 0

Rows and cols after cleaning: (290584, 5)


,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1


## Conversion

In [8]:
#Average per group
df_conversion = (df.groupby('group')
                 .agg({
                     'converted':"mean"
                 }))

print(f"    Current conversion per group:{df_conversion}\n")

print(f"    Current users per group:{df.group.value_counts()}\n")



    Current conversion per group:           converted
group               
control     0.120386
treatment   0.118808

    Current users per group:group
treatment    145310
control      145274
Name: count, dtype: int64



Difference is only 0,16%. It could be a random result

## Power analysis

In [22]:
effect_size = proportion_effectsize((df_conversion.loc['control','converted']), (df_conversion.loc['control','converted']+0.02))
analysis = NormalIndPower()
sample_size = analysis.solve_power(
    effect_size=effect_size,
    power=0.80,
    alpha=0.05,
    alternative='two-sided'
)

#Vars
converted_control = df[df['group'] == 'control']['converted'].sum()
converted_treatment = df[df['group'] == 'treatment']['converted'].sum()
total_control = df[df['group'] == 'control']['converted'].count()
total_treatment = df[df['group'] == 'treatment']['converted'].count()


print(f"N Necessary per group: {sample_size:,.0f}: \nCurrent users in control group: {total_control:,} \nCurrent users in treatment group: {total_treatment:,}")

print(f"Current converted users in control group: {converted_control:,} \nCurrent converted users in treatment group: {converted_treatment:,}")



N Necessary per group: 4,444: 
Current users in control group: 145,274 
Current users in treatment group: 145,310
Current converted users in control group: 17,489 
Current converted users in treatment group: 17,264


### Z-Test

In [27]:
conversions = [converted_control,converted_treatment]
nobs = [total_control,total_treatment]

stat, p_value = proportions_ztest(conversions, nobs)
print(f"Z-statistic: {stat:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < umbral:
    print("Significant difference")
else:
    print ("insignificant difference")


Z-statistic: 1.3109
P-value: 0.1899
insignificant difference


There is no significant evidence to affirm that the new page is better than the old one

Business desicion: not worth implementing

#### Summary
- Defined H₀ and H₁ before looking at the data.
- Calculated the required sample size before running the test.
- Used a single primary metric — conversion rate.
- Stopped when the target sample size was reached, not when the result was favorable.